# Ludwig 2 · T0.3 + T2: learning rate y estabilidad en FP16, en 2×T4

Entrena el **modelo real** (1.906B totales, 0.491B activos) con `scripts/train.py`, el bucle de entrenamiento de verdad: el lector de datos con reanudación exacta, el schedule WSD, la evaluación en bits por byte, las alarmas del diseño (5.4) y los checkpoints con reanudación verificada. Usa las piezas que eligió la T0.2 v2: un matmul por experto, residuo en FP32, gradientes en FP16 entre las GPUs, prefetch y el momento de Muon en la RAM de la CPU.

**Son tres corridas iguales que solo cambian el learning rate**, una por cuenta y en paralelo:

| Cuenta | `CORRIDA` | `LR_MULT` | Muon / AdamW |
|---|---|---|---|
| cuenta 1 | `lr-x0.5` | 0.5 | 7.5e-4 / 2.5e-4 |
| cuenta 2 | `lr-x1` | 1.0 | 1.5e-3 / 5e-4 |
| cuenta 3 | `lr-x2` | 2.0 | 3e-3 / 1e-3 |

**×1 es la receta del Ludwig 1:** AdamW 5e-4 para embeddings, normas y router; Muon 3 veces eso, 1.5e-3 (en el A/B del 31 de octubre el ×3 le ganó a AdamW por 0.37 nats); weight decay 0.1 aplicado con el LR de AdamW también a las matrices de Muon. Los dos Muon usan la misma escala (la de Moonlight), así que los números se pueden comparar directamente.

Cada una hace **400 pasos de 0.25M tokens (~105M tokens)**: 100 de calentamiento, LR constante y un enfriamiento 1−√ en los últimos 80 pasos (el último 20%, como en el Ludwig 1), con el mismo orden de datos. Se comparan los bits por byte de validación al final. La de ×2 es además la prueba de estabilidad en FP16 con margen.

**Datos y tokenizador provisionales:** FineWeb-Edu (`sample-10BT`, archivos 000 y 001 para entrenar y 013 para validar) y el tokenizador de SmolLM2 (49,152 tokens, Apache 2.0, con `<|endoftext|>` = 0 como el nuestro), mientras se termina el de 64K. Por eso la métrica son los bits por byte, que no dependen del tokenizador.

**Kaggle:** acelerador *GPU T4 x2*, Internet *On*, el repo agregado como dataset. *Save Version → Save & Run All*. Tarda **~9.5 h** de las 12 que da la sesión; si se acaba el tiempo, guarda un checkpoint y para limpio.

**Segunda ronda (regla P9 de `docs/DECISIONS.md`).** En la primera ganó ×2. Las corridas cortas favorecen los LR altos, así que la segunda mide cuánto se mueve el mejor LR cuando la corrida es más larga:

| Cuenta | `CORRIDA` | `LR_MULT` | `LAMBDA` | `PASOS, CALENTAMIENTO, ENFRIAMIENTO` | `REANUDAR` | `PRUEBA` | Sesiones |
|---|---|---|---|---|---|---|---|
| una libre | `lr-x4` | 4.0 | 0.04 | 400, 100, 80 | `""` | `True` | 1 |
| la que corrió `lr-x1` | `lr-x1-1200` | 1.0 | 0.01 | 1200, 100, 240 | `"auto"` | `False` | 2 |
| la que corrió `lr-x2` | `lr-x2-1200` | 2.0 | 0.01 | 1200, 100, 240 | `"auto"` | `False` | 2 |

- `lr-x4` cierra el barrido corto, con el balanceo de SMEBU escalado con el LR (λ = 0.01 × el multiplicador).
- Las otras dos **siguen** las corridas ×1 y ×2 desde su checkpoint del paso 320 hasta 1,200 pasos, 3 veces más largas, con el mismo λ que tuvieron.
- Para seguir una corrida, agrega como entrada (*Add Input → Your Work*) la salida de la versión que la corrió. Con `REANUDAR = "auto"` el notebook busca ahí el checkpoint de esa corrida más avanzado.
- La primera sesión para sola hacia el paso 800 y deja su checkpoint en la salida. La segunda se lanza igual, con la salida de la primera como entrada, y termina en el 1,200.

In [ ]:
CORRIDA = "lr-x1"      # lr-x0.5 · lr-x1 · lr-x2: una por cuenta
LR_MULT = 1.0          # 0.5 · 1.0 · 2.0
REPO_URL = ""          # "" = el repo subido como dataset de Kaggle (el repo es privado)
PASOS, CALENTAMIENTO, ENFRIAMIENTO = 400, 100, 80
LAMBDA = 0.01          # SMEBU: 0.01 × LR_MULT en una corrida nueva; el mismo de antes en una que sigue
GUARDAR_CHECKPOINT = True   # deja en la salida el checkpoint de antes del enfriamiento (~11.3 GB), para seguir la corrida
REANUDAR = ""          # para seguir una corrida: "auto" (lo busca en los datos de entrada) o la carpeta del checkpoint
PRUEBA = True          # la prueba de guardado y reanudación (~5 min); al seguir una corrida sobra: se verifica al reanudar
import time
T_SESION = time.time()

In [ ]:
import os, subprocess, sys, json
os.chdir("/kaggle/working")
if not os.path.exists("ludwig-lm"):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "ludwig-lm"], check=True)
    else:  # el repo subido como dataset de Kaggle: se busca la carpeta que contiene ludwig_lm/
        import glob
        src = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/ludwig_lm/__init__.py", recursive=True)[0]))
        subprocess.run(["cp", "-r", src, "ludwig-lm"], check=True)
os.chdir("/kaggle/working/ludwig-lm")
!pip install -q "transformers==5.22.2" safetensors sentencepiece protobuf
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CPUs", os.cpu_count(),
      "| GPUs", torch.cuda.device_count())
!pip install -q hf_xet
!nvidia-smi --query-gpu=name,memory.total,power.limit --format=csv
!df -h /tmp /kaggle/working | tail -2
!free -g | head -2

## 1. Datos y tokenizador

Se bajan de Hugging Face sin token (son públicos): ~4.8 GB.

In [ ]:
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq
t = time.time()
DATOS = "/tmp/ludwig_datos"
TRAIN = [hf_hub_download("HuggingFaceFW/fineweb-edu", f"sample/10BT/{i:03d}_00000.parquet", repo_type="dataset",
                         local_dir=DATOS) for i in (0, 1)]
VAL = hf_hub_download("HuggingFaceFW/fineweb-edu", "sample/10BT/013_00000.parquet", repo_type="dataset", local_dir=DATOS)
TOK = hf_hub_download("HuggingFaceTB/SmolLM2-135M", "tokenizer.json", local_dir="/tmp/ludwig_tok")
for f in TRAIN + [VAL]:
    m = pq.ParquetFile(f).metadata
    print(f"{os.path.basename(f)}: {os.path.getsize(f) / 1e9:.2f} GB, {m.num_rows:,} documentos, {m.num_row_groups} row groups")
print(f"listo en {time.time() - t:.0f} s")
COMUN = dict(tokenizer=TOK, train=" ".join(TRAIN), val=VAL)

In [ ]:
import threading, time, glob, shutil, re
DUENO = re.compile(r"(/kaggle/input/(?:notebooks|datasets|models)/)[^/\"\s]+")
publico = lambda texto: DUENO.sub(r"\1<usuario>", texto)   # el usuario de Kaggle de una entrada no va a los logs
KEEP = ("[model]", "[data]", "[train]", "[eval]", "[save]", "[resume]", "[alarm]", "[end]", "[checkpoint]",
        "Traceback", "Error", "out of memory")

def run(tag, out, timeout_h, **kw):
    """Lanza scripts/train.py con torchrun en las 2 GPUs, muestra el progreso y devuelve su fila final."""
    def arg(k, v):
        flag = "--" + k.replace("_", "-")
        return flag if v is True else f"{flag} {v}"
    args = " ".join(arg(k, v) for k, v in kw.items() if v is not False and v != "")
    cmd = (f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True torchrun --nproc_per_node 2 "
           f"scripts/train.py --tag {tag} --out {out} {args}")
    print(">>", publico(cmd), flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         start_new_session=True)
    killed = []
    def kill():  # último recurso: train.py se detiene solo con --max-hours
        killed.append(True)
        subprocess.run(["pkill", "-TERM", "-f", "scripts/train.py"])
        time.sleep(30)
        subprocess.run(["pkill", "-KILL", "-f", "scripts/train.py"])
    timer = threading.Timer(timeout_h * 3600, kill)
    timer.start()
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if any(k in line for k in KEEP):
            print(publico(line), end="", flush=True)
    p.wait()
    timer.cancel()
    if p.returncode != 0:
        print(f"[{tag}] terminó con código {p.returncode}" + (" (se pasó del tiempo)" if killed else "")
              + "; últimas líneas:\n" + publico("".join(tail[-15:])))
    rows = [json.loads(l) for l in open(out)] if os.path.exists(out) else []
    return next((r for r in reversed(rows) if r.get("tag") == tag and r.get("type") == "end"),
                {"tag": tag, "status": "sin fila final"})

## 2. Prueba de guardado y reanudación (~5 min)

Dos pasos cortos, guardar, y reanudar en otro proceso. La reanudación compara el estado cargado bit a bit con el que se guardó (pesos, momento de Muon en la CPU, AdamW, bias de los expertos) y el primer lote con el hash guardado: lo que pide la G0. Si algo falla, la corrida del paso 3 sigue igual: solo su checkpoint quedaría en duda.

In [ ]:
if PRUEBA:
    ARCHIVO_PRUEBA = f"results/t0_3_t2/{CORRIDA}-prueba.jsonl"
    CORTA = dict(**COMUN, warmup=1, tokens_per_step=32768, eval_every=0, eval_windows=8)
    run("prueba-a", ARCHIVO_PRUEBA, 0.5, **CORTA, steps=2, save_dir="/tmp/ludwig_prueba", save_final=True)
    run("prueba-b", ARCHIVO_PRUEBA, 0.5, **CORTA, steps=3, resume="/tmp/ludwig_prueba/step_000002")
    filas = [r for r in map(json.loads, open(ARCHIVO_PRUEBA)) if r["time"] >= T_SESION]   # solo esta sesión
    res = next((r for r in filas if r["type"] == "resume"), {})
    lote = next((r for r in filas if r["type"] == "resume_check"), {})
    ev = {(r["tag"], r["step"]): r["val_bpb"] for r in filas if r["type"] == "eval"}
    dif = abs(ev.get(("prueba-a", 2), 0) - ev.get(("prueba-b", 2), 9))
    print(f"estado idéntico: {res.get('state_exact')} | primer lote idéntico: {lote.get('batch_exact')} | "
          f"validación antes/después de reanudar: {dif:.2e} bits por byte de diferencia")
    print("✅ guardado y reanudación exactos" if res.get("state_exact") and lote.get("batch_exact") and dif < 1e-3
          else "⚠️ la reanudación no salió exacta: revisar antes de confiar en los checkpoints")
    shutil.rmtree("/tmp/ludwig_prueba", ignore_errors=True)
else:
    print("sin prueba: esta corrida sigue un checkpoint, y su reanudación se verifica al cargarlo")

## 3. La corrida

Imprime una línea cada 10 pasos y una evaluación cada 50 (más una antes del enfriamiento y otra al final). Cada paso queda en `results/t0_3_t2/<CORRIDA>.jsonl` apenas termina. Se detiene sola antes de las 12 h de la sesión, guardando un checkpoint.

In [ ]:
SALIDA = f"results/t0_3_t2/{CORRIDA}.jsonl"
horas = round(11.3 - (time.time() - T_SESION) / 3600, 2)
kw = dict(**COMUN, steps=PASOS, warmup=CALENTAMIENTO, decay_steps=ENFRIAMIENTO, lr_mult=LR_MULT, lam=LAMBDA,
          eval_every=50, save_dir="/tmp/ludwig_ckpt", max_hours=horas)
if REANUDAR == "auto":
    # los checkpoints de esta corrida, o de la que sigue ("lr-x1" para "lr-x1-1200"), en los datos de entrada
    hay = []
    for m in glob.glob("/kaggle/input/**/checkpoint/meta.json", recursive=True):
        meta = json.load(open(m))
        if meta["tag"] == CORRIDA or CORRIDA.startswith(meta["tag"] + "-"):
            hay.append((meta["step"], os.path.dirname(m), meta["tag"], meta["reason"]))
    if not hay:
        raise SystemExit(f"no hay checkpoint de {CORRIDA} en los datos de entrada: agrega la salida de la versión que la corrió")
    paso, REANUDAR, de, motivo = max(hay)
    print(f"sigue {de} desde el paso {paso} ({motivo}): {publico(REANUDAR)}")
    # si es la segunda sesión de esta misma corrida, su .jsonl sigue en el mismo archivo
    antes = glob.glob(os.path.join(os.path.dirname(REANUDAR), "**", "results", "t0_3_t2", f"{CORRIDA}.jsonl"),
                      recursive=True)
    if antes and not os.path.exists(SALIDA):
        shutil.copy(antes[0], SALIDA)
        print(f"{SALIDA}: sigue el de la sesión anterior ({sum(1 for _ in open(SALIDA))} filas)")
if REANUDAR:
    kw["resume"] = REANUDAR
print(f"{CORRIDA}: LR ×{LR_MULT}, λ {LAMBDA}, {PASOS} pasos, tope {horas} h")
fin = run(CORRIDA, SALIDA, horas + 0.3, **kw)
print(fin)

## 4. Resumen

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
filas = [r for r in map(json.loads, open(SALIDA)) if r["tag"] == CORRIDA and r["time"] >= T_SESION]
pasos = pd.DataFrame([r for r in filas if r["type"] == "step"])
evals = pd.DataFrame([r for r in filas if r["type"] == "eval"])
fin = ([r for r in filas if r["type"] == "end"] or [{}])[-1]
if len(evals):
    display(evals[["step", "tokens", "val_bpb", "val_ce", "router_entropy", "qk_gain_max", "dead_experts_window"]])
if fin.get("status") in (None, "error"):
    print("⚠️ la corrida no terminó bien: ver el log de arriba", fin.get("error", ""))
else:
    n = lambda x: "—" if x is None else f"{x:,}"
    print(f"{CORRIDA}: {fin['status']}, paso {fin['steps_done']}, {fin['tokens'] / 1e6:.0f}M tokens en "
          f"{fin['hours_this_session']:.2f} h")
    print(f"  val bpb antes del enfriamiento {fin['val_bpb_pre_decay']} -> al final {fin['val_bpb_final']}")
    print(f"  FP16: {fin['loss_scale_skips']} pasos saltados, escala final {fin['loss_scale_last']:.0f}; "
          f"picos de pérdida saltados: {fin['spikes_skipped']}; alarmas: {fin['alarms']}")
    tps, pico = fin["tokens_per_s_last_hour"], fin["peak_reserved_GiB"]
    print(f"  velocidad sostenida (última hora): {n(tps)} tokens/s, MFU {fin['mfu']}, ~{n(fin['hours_for_25B'])} h "
          f"para 25B | G0 pide ≥ 2,600: {'✅' if tps and tps >= 2600 else '❌'}")
    print(f"  memoria pico: {pico} GiB reservada, {fin['peak_alloc_GiB']} asignada | G0 pide ≤ 13.5 reservada: "
          f"{'✅' if pico <= 13.5 else '❌'}")
for r in filas:
    if r["type"] == "alarm":
        print("  alarma:", r["step"], r["alarm"])
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(pasos.tokens / 1e6, pasos.ce.rolling(10, min_periods=1).mean())
ax[0].set(xlabel="millones de tokens", ylabel="nats por token", title=f"{CORRIDA}: pérdida de entrenamiento (media de 10 pasos)")
ax[1].plot(evals.tokens / 1e6, evals.val_bpb, "o-")
ax[1].set(xlabel="millones de tokens", ylabel="bits por byte", title="validación",
          ylim=(None, evals.val_bpb.iloc[1:].max() * 1.05) if len(evals) > 1 else (None, None))
ax[2].plot(pasos.step, pasos.load_max_over_mean.rolling(10, min_periods=1).mean())
ax[2].axhline(3, ls="--", c="gray")
ax[2].set(xlabel="paso", ylabel="experto más cargado / promedio", title="balanceo (media de 10 pasos; alarma en 3)")
fig.tight_layout()
fig.savefig(f"results/t0_3_t2/{CORRIDA}.png", dpi=120)
plt.show()

In [ ]:
# El checkpoint de antes del enfriamiento (paso 320) a la salida del notebook, para poder seguir esta corrida
if GUARDAR_CHECKPOINT:
    hechos = sorted(os.path.dirname(m) for m in glob.glob("/tmp/ludwig_ckpt/step_*/meta.json"))
    pre = [d for d in hechos if json.load(open(f"{d}/meta.json"))["reason"] == "pre_decay"]
    elegido = (pre or hechos or [None])[-1]
    if elegido:
        shutil.rmtree("/kaggle/working/checkpoint", ignore_errors=True)
        shutil.copytree(elegido, "/kaggle/working/checkpoint")
        print("checkpoint en la salida:", elegido, "->", "/kaggle/working/checkpoint")
    else:
        print("no hay checkpoint que guardar")
!du -sh /kaggle/working/checkpoint 2>/dev/null; ls results/t0_3_t2/

## Qué hacer con el resultado

1. Descarga de la salida del notebook `results/t0_3_t2/` (el `.jsonl` de la corrida, el de la prueba y el `.png`) y el log de Kaggle, guárdalos en `results/t0_3_t2/` y anota la entrada en la bitácora.
2. **Regla de decisión, fijada antes de ver los resultados:**
   - Una corrida cuenta solo si fue estable: sin NaN, sin parar por picos y con la escala de FP16 estable.
   - Gana la de menor bits por byte de validación **al final** (después del enfriamiento). Si dos quedan a menos de 0.5%, gana la de LR más bajo: las corridas cortas favorecen los LR altos, y el tronco es 200 veces más largo.
   - Si gana un extremo (×0.5 o ×2), se corre uno más allá (×0.25 o ×4) antes de decidir.
   - SMEBU con λ = 0.01 queda si la carga máx/media se mantiene bajo 3 y no hay expertos muertos.
3. **Segunda ronda (P9):** con los `.jsonl` de `lr-x4`, `lr-x1-1200` y `lr-x2-1200` en `results/t0_3_t2/`, `python scripts/lr_horizon.py` da el LR del tronco. Mide cuánto baja el mejor LR de 400 a 1,200 pasos y lo lleva a los 25B tokens del tronco.
4. **Para seguir una corrida:** agrega como entrada la salida de la versión que la corrió y pon `REANUDAR = "auto"`. Sube `PASOS` y pon el `ENFRIAMIENTO` que toque. La corrida sigue con el mismo orden de datos, desde el checkpoint más avanzado de esa corrida que encuentre.